# Federated Learning Labs 1–4 · Complete Solutions
**Course:** Fundamentals & Architectures of Federated Learning
**Lecturer:** Ts. Dr. Mohammad Kamrul Hasan, Centre for Cyber Security, FTSM, Universiti Kebangsaan Malaysia (UKM)

> Uses Flower's current **Message API**, checked against Flower 1.44. Runs on CPU, but a GPU is strongly recommended: *Runtime → Change runtime type → GPU*.

All four lab steps in one notebook: (1) quickstart with 10 clients, (2) Dirichlet α = 0.1, (3) FedAvg vs FedProx, (4) DP clipping. Each lab reuses the same setup cells.

## Setup 1 · Install Flower

In [ ]:
!pip install -q "flwr[simulation]==1.44.0" "flwr-datasets[vision]==0.6.1"
import flwr, flwr_datasets
print("Flower", flwr.__version__, "| Flower Datasets", flwr_datasets.__version__)

## Setup 2 · Imports and settings
K = 10 clients, E = 1 local epoch, B = 32, η = 0.01. `NUM_ROUNDS` = 30 as in the lab sheet; set `QUICK_MODE = True` for a fast first try.

In [ ]:
import copy, random
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Normalize, ToTensor

from flwr.app import ArrayRecord, ConfigRecord, Context, Message, MetricRecord, RecordDict
from flwr.clientapp import ClientApp
from flwr.clientapp.mod import fixedclipping_mod
from flwr.serverapp import Grid, ServerApp
from flwr.serverapp.strategy import FedAvg, FedProx, DifferentialPrivacyClientSideFixedClipping
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner, DirichletPartitioner

NUM_CLIENTS   = 10      # K
LOCAL_EPOCHS  = 1       # E
BATCH_SIZE    = 32      # B
LEARNING_RATE = 0.01    # client η
QUICK_MODE    = False   # True = 10 rounds instead of 30 (for a fast first try)
NUM_ROUNDS    = 10 if QUICK_MODE else 30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on", DEVICE, "| rounds per experiment:", NUM_ROUNDS)
torch.manual_seed(0); np.random.seed(0); random.seed(0)

## Setup 3 · Data: CIFAR-10 split across 10 clients
`make_partitioner` chooses an **IID** split or a non-IID **Dirichlet(α)** split. Each simulated client loads only its own partition. The server keeps the official test set for evaluation.

In [ ]:
_transforms = Compose([ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
def apply_transforms(batch):
    batch["img"] = [_transforms(img) for img in batch["img"]]
    return batch

def make_partitioner(partition: str, alpha: float, num_partitions: int):
    """'iid' = every client gets a similar class mix; 'dirichlet' = non-IID, controlled by alpha."""
    if partition == "iid":
        return IidPartitioner(num_partitions=num_partitions)
    return DirichletPartitioner(num_partitions=num_partitions, partition_by="label",
                                alpha=alpha, min_partition_size=10, seed=42)

_CLIENT_FDS = {}   # per-process cache, filled only inside the simulated clients
def load_client_data(partition_id, num_partitions, partition, alpha):
    key = (partition, alpha, num_partitions)
    if key not in _CLIENT_FDS:
        _CLIENT_FDS[key] = FederatedDataset(dataset="uoft-cs/cifar10",
            partitioners={"train": make_partitioner(partition, alpha, num_partitions)})
    data = _CLIENT_FDS[key].load_partition(partition_id).with_transform(apply_transforms)
    return DataLoader(data, batch_size=BATCH_SIZE, shuffle=True)

def load_test_set():
    """The server's held-out CIFAR-10 test set (10,000 images)."""
    test = FederatedDataset(dataset="uoft-cs/cifar10", partitioners={"train": 1}).load_split("test")
    return DataLoader(test.with_transform(apply_transforms), batch_size=256)

def show_partition(partition="iid", alpha=None):
    """Plot how the 10 clients' label mixes look (runs in the notebook, not in the clients)."""
    from flwr_datasets.visualization import plot_label_distributions
    fds = FederatedDataset(dataset="uoft-cs/cifar10",
                           partitioners={"train": make_partitioner(partition, alpha, NUM_CLIENTS)})
    p = fds.partitioners["train"]
    print("Images per client:", [len(p.load_partition(i)) for i in range(NUM_CLIENTS)])
    title = "IID" if partition == "iid" else f"Dirichlet α = {alpha}"
    plot_label_distributions(p, label_name="label", plot_type="bar", size_unit="percent",
                             legend=True, title=f"Label mix per client ({title})")
    plt.show()

## Setup 4 · Model, local training (with optional FedProx term) and testing

In [ ]:
class Net(nn.Module):
    """Small CNN from the Flower PyTorch quickstart."""
    def __init__(self):
        super().__init__()
        self.conv1, self.pool, self.conv2 = nn.Conv2d(3, 6, 5), nn.MaxPool2d(2, 2), nn.Conv2d(6, 16, 5)
        self.fc1, self.fc2, self.fc3 = nn.Linear(400, 120), nn.Linear(120, 84), nn.Linear(84, 10)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))); x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 400); x = F.relu(self.fc1(x)); x = F.relu(self.fc2(x))
        return self.fc3(x)

def train(net, loader, epochs, lr, mu=0.0):
    """Local SGD. If mu > 0, adds the FedProx term (mu/2)·||w − w_global||²."""
    net.to(DEVICE).train()
    global_params = [p.detach().clone() for p in net.parameters()]   # w^t, before training
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    total, steps = 0.0, 0
    for _ in range(epochs):
        for batch in loader:
            x, y = batch["img"].to(DEVICE), batch["label"].to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(net(x), y)
            if mu > 0:
                prox = sum(((w - g) ** 2).sum() for w, g in zip(net.parameters(), global_params))
                loss = loss + (mu / 2) * prox
            loss.backward(); opt.step()
            total += loss.item(); steps += 1
    return total / max(steps, 1)

def test(net, loader):
    net.to(DEVICE).eval(); correct, loss = 0, 0.0
    with torch.no_grad():
        for batch in loader:
            x, y = batch["img"].to(DEVICE), batch["label"].to(DEVICE)
            out = net(x); loss += F.cross_entropy(out, y).item()
            correct += (out.argmax(1) == y).sum().item()
    return loss / len(loader), correct / len(loader.dataset)

## Setup 5 · Client code
The server sends the learning rate and the partition settings in the train config. FedProx adds `proximal-mu` to that config automatically. `dp_client_app` is the same client with `fixedclipping_mod`, which clips every update before it leaves the device (Lab 4).

In [ ]:
def client_train(msg: Message, context: Context):
    """ClientUpdate: load w^t, train on this client's own data, reply with w_k and n_k."""
    cfg = msg.content["config"]                       # settings sent by the server
    model = Net()
    model.load_state_dict(msg.content["arrays"].to_torch_state_dict())
    loader = load_client_data(context.node_config["partition-id"],
                              context.node_config["num-partitions"],
                              cfg["partition"], cfg["alpha"])
    mu = cfg.get("proximal-mu", 0.0)                  # added automatically by FedProx
    loss = train(model, loader, LOCAL_EPOCHS, cfg["lr"], mu)
    return Message(content=RecordDict({
        "arrays": ArrayRecord(torch_state_dict=model.state_dict()),
        "metrics": MetricRecord({"train_loss": loss, "num-examples": len(loader.dataset)})}),
        reply_to=msg)

client_app = ClientApp();                             client_app.train()(client_train)
dp_client_app = ClientApp(mods=[fixedclipping_mod]);  dp_client_app.train()(client_train)   # clips each update

## Setup 6 · `run_experiment` and plotting helpers
Each call builds a ServerApp with the chosen strategy (FedAvg, FedProx, optionally wrapped with DP), runs 10 simulated clients and records global test accuracy after every round.

In [ ]:
RESULTS = {}   # name -> list of global test accuracy (%) per round

def run_experiment(name, partition="iid", alpha=1.0, mu=0.0, dp_noise=None, clip=1.0,
                   rounds=None, fraction=1.0):
    """One federated training run. Returns the accuracy (%) after each round."""
    rounds = rounds or NUM_ROUNDS
    m = max(1, int(fraction * NUM_CLIENTS))          # clients sampled per round
    server_app = ServerApp()

    @server_app.main()
    def main(grid: Grid, context: Context) -> None:
        testloader = load_test_set()
        def central_evaluate(server_round, arrays):
            net = Net(); net.load_state_dict(arrays.to_torch_state_dict())
            loss, acc = test(net, testloader)
            return MetricRecord({"loss": loss, "accuracy": acc})
        base = (FedProx(fraction_train=fraction, fraction_evaluate=0.0, proximal_mu=mu)
                if mu > 0 else FedAvg(fraction_train=fraction, fraction_evaluate=0.0))
        strategy = base if dp_noise is None else DifferentialPrivacyClientSideFixedClipping(
            base, noise_multiplier=dp_noise, clipping_norm=clip, num_sampled_clients=m)
        result = strategy.start(grid=grid, num_rounds=rounds,
            initial_arrays=ArrayRecord(torch_state_dict=Net().state_dict()),
            train_config=ConfigRecord({"lr": LEARNING_RATE, "partition": partition,
                                       "alpha": float(alpha)}),
            evaluate_fn=central_evaluate)
        hist = result.evaluate_metrics_serverapp
        RESULTS[name] = [hist[r]["accuracy"] * 100 for r in sorted(hist) if r > 0]

    run_simulation(server_app=server_app,
                   client_app=client_app if dp_noise is None else dp_client_app,
                   num_supernodes=NUM_CLIENTS,
                   backend_config={"client_resources": {"num_cpus": 1,
                                   "num_gpus": 0.1 if DEVICE.type == "cuda" else 0.0}})
    print(f"\n{name}: final accuracy {RESULTS[name][-1]:.1f}%")
    return RESULTS[name]

STYLES = [("#2B5BA8", "--", "o"), ("#A2472F", "-.", "^"), ("#2F6E62", "-", "s"),
          ("#D49A3A", ":", "D"), ("#5E6672", (0, (5, 1)), "v")]
def plot_runs(names, title):
    plt.figure(figsize=(9, 4.5))
    for (c, ls, mk), n in zip(STYLES, names):
        y = RESULTS[n]; plt.plot(range(1, len(y) + 1), y, color=c, linestyle=ls, marker=mk,
                                 markersize=4, label=f"{n} ({y[-1]:.1f}%)")
    plt.xlabel("Communication round"); plt.ylabel("Global test accuracy (%)")
    plt.title(title); plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

# Lab 1 · Run the Flower PyTorch quickstart with 10 clients
FedAvg on an IID split, all 10 clients each round. Watch the log: every round shows `configure_train: Sampled 10 nodes (out of 10)` and `aggregate_train: Received 10 results and 0 failures`.

In [ ]:
show_partition("iid")

In [ ]:
run_experiment("FedAvg, IID", partition="iid")
plot_runs(["FedAvg, IID"], "Lab 1: FedAvg with 10 clients (IID)")

**Report:** Flower version, clients, rounds, epochs, learning rate, final accuracy and the plot. **Expected:** accuracy rises over the first rounds and then levels off.

# Lab 2 · Switch to a Dirichlet α = 0.1 partition
Only the partitioner changes. With α = 0.1 most clients hold one or two dominant classes (non-IID = not independent and identically distributed).

In [ ]:
show_partition("dirichlet", alpha=0.1)

In [ ]:
run_experiment("FedAvg, IID", partition="iid")
run_experiment("FedAvg, Dirichlet α=0.1", partition="dirichlet", alpha=0.1)
plot_runs(["FedAvg, IID", "FedAvg, Dirichlet α=0.1"], "Lab 2: IID vs non-IID (FedAvg)")

**Expected:** the non-IID run learns more slowly, oscillates more and ends lower, because each client's update pulls towards its own few classes (client drift). In our reference run on a simple model the drop was 82.0% → 77.9%; your CIFAR-10 numbers will differ but the gap should be clear.

**Report:** the label-distribution plot, both curves, and the final accuracy gap.

# Lab 3 · Compare FedAvg and FedProx (μ = 0.1, 0.5) over 30 rounds
All three runs use the Dirichlet α = 0.1 split. FedProx adds `(μ/2)·‖w − wᵗ‖²` to every client's loss, pulling local models back towards the global model. μ = 0 is plain FedAvg.

In [ ]:
for name, mu in [("FedAvg (μ = 0)", 0.0), ("FedProx μ = 0.1", 0.1), ("FedProx μ = 0.5", 0.5)]:
    run_experiment(name, partition="dirichlet", alpha=0.1, mu=mu)
plot_runs(["FedAvg (μ = 0)", "FedProx μ = 0.1", "FedProx μ = 0.5"], "Lab 3: FedAvg vs FedProx on Dirichlet α = 0.1")

In [ ]:
print(f"{'Setting':<18}{'Round 5':>10}{'Round 30':>10}{'Best':>10}")
for n in ["FedAvg (μ = 0)", "FedProx μ = 0.1", "FedProx μ = 0.5"]:
    y = RESULTS[n]
    print(f"{n:<18}{y[min(4, len(y)-1)]:>9.1f}%{y[-1]:>9.1f}%{max(y):>9.1f}%")

**Expected:** FedProx curves are smoother and end at or above FedAvg; in our reference run μ = 0.5 closed most of the gap (77.9% → 81.5%). Too large a μ can slow learning, so report the μ you would choose and why.

**Report:** one plot with all three curves, and the table (round 5, round 30, best).

# Lab 4 · Add DP clipping and report the accuracy cost
Central differential privacy with client-side clipping: every client clips its update to norm **C** (`fixedclipping_mod`), and the server adds Gaussian noise with standard deviation **σ·C / m** (m = clients per round). We run four settings on the Dirichlet α = 0.1 split:
- no DP (baseline)
- clipping only (C = 1, σ = 0) — bounds each client's influence, no formal privacy yet
- C = 1, σ = 0.5
- C = 1, σ = 1.0

In [ ]:
run_experiment("No DP", partition="dirichlet", alpha=0.1)
run_experiment("Clip only (C=1, σ=0)", partition="dirichlet", alpha=0.1, dp_noise=0.0, clip=1.0)
run_experiment("C=1, σ=0.5", partition="dirichlet", alpha=0.1, dp_noise=0.5, clip=1.0)
run_experiment("C=1, σ=1.0", partition="dirichlet", alpha=0.1, dp_noise=1.0, clip=1.0)

In [ ]:
names = ["No DP", "Clip only (C=1, σ=0)", "C=1, σ=0.5", "C=1, σ=1.0"]
base = RESULTS["No DP"][-1]
print(f"{'Run':<22}{'Accuracy':>10}{'Cost':>12}")
for n in names:
    a = RESULTS[n][-1]; print(f"{n:<22}{a:>9.1f}%{(a - base):>+10.1f} pts")
plot_runs(names, "Lab 4: accuracy cost of DP (Dirichlet α = 0.1)")
plt.figure(figsize=(7, 3.5))
plt.bar(names, [RESULTS[n][-1] for n in names], color=["#1A2433", "#2F6E62", "#D49A3A", "#A2472F"])
plt.ylabel("Final accuracy (%)"); plt.title("Final accuracy per DP setting"); plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

**Expected:** clipping alone costs a little; adding noise costs more, and the cost grows quickly with σ. In our reference run: no DP 77.9%, clip only 74.2%, σ = 0.5 69.7%, σ = 1.0 57.4%.

**Why so expensive here?** The noise std is σ·C/m, and m is only 10. With m = 1,000 clients per round the same σ would add 100× less noise per coordinate, which is why DP works well in large cross-device deployments.

**Report:** a table of C, σ, m and final accuracy, the plots, and one sentence on the privacy–accuracy trade-off. *(To report ε, Flower's DP wrappers accept a `PrivacyAccountant`; see the Flower docs.)*

## Troubleshooting
| Symptom | Fix |
|---|---|
| `Received 0 results and N failures` | A client crashed: scroll up to the client traceback |
| Ray / out-of-memory errors | Use a GPU runtime, or lower `num_cpus` / `num_gpus` per client |
| Very slow on CPU | Set `QUICK_MODE = True` (10 rounds) for a first pass |
| First run is slow | CIFAR-10 is downloading; it is cached afterwards |
| `run_simulation` deprecation note | Expected; it still works and is the simplest option in a notebook |